# SmokeShift on Google Colab

Runs the GPU-heavy stages of [SmokeShift](https://github.com/HameesSajid/smokeshift) on a Colab GPU and packs the
results so they can be copied back into the project.

**Before running:** Runtime > Change runtime type > pick a GPU (T4 is enough; L4 or A100 is faster).

What this notebook does:

1. checks the GPU
2. gets the code (and the 17 MB cleaned panel, which is in the repository, so no EPA download is needed)
3. installs the two foundation-model packages
4. runs the stages you choose
5. zips the tables, figures, report and dashboard for download

Stored forecasts are large (about 0.7 GB per model). They are kept on your Google Drive if you mount it in step 2,
otherwise they disappear when the session ends, which is fine if you only want the tables and figures.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Get the code

If the repository is **public**, the first cell works as it is.

If it is **private**, create a GitHub personal access token with read access to the repository, add it in Colab
under the key icon (Secrets) with the name `GITHUB_TOKEN`, and allow this notebook to read it. The token never
appears in the notebook.

In [ ]:
import os, subprocess
REPO = "HameesSajid/smokeshift"
url = f"https://github.com/{REPO}.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    pass
if not os.path.exists("/content/smokeshift"):
    result = subprocess.run(["git", "clone", "--depth", "1", url, "/content/smokeshift"], capture_output=True, text=True)
    print("cloned" if result.returncode == 0 else "clone failed: is the repository private and GITHUB_TOKEN missing?")
%cd /content/smokeshift
!git log --oneline -1

## 2. Where to keep large files

Run **one** of the next two cells. The first keeps forecasts and model weights on Google Drive so they survive the
session. The second keeps them on the Colab disk only.

In [ ]:
# Option A: keep forecasts and weights on Google Drive (about 12 GB for all models)
from google.colab import drive
drive.mount("/content/drive")
os.environ["SMOKESHIFT_CACHE"] = "/content/drive/MyDrive/smokeshift_cache"

In [ ]:
# Option B: temporary Colab disk
os.environ["SMOKESHIFT_CACHE"] = "/content/smokeshift_cache"

## 3. Install the extra packages

Colab already has PyTorch with CUDA, XGBoost and the scientific stack.

In [ ]:
!pip -q install chronos-forecasting "timesfm[torch]"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
!python tests/test_smokeshift.py

## 4. Run

Forecasts are not in the repository, so every model you want in the comparison has to be produced here. The cells
below are in dependency order. On the laptop's GTX 1660 Ti the whole section takes four to five hours, most of it
the six foundation-model variants. Colab GPUs are faster.

In [ ]:
# Reference forecasts and the three tree models (GPU). Tuning is skipped: the chosen settings are in
# outputs/tables/gbm_tuning.json and are reused. Add "tune" before "gbm" to search again (30 minutes).
!python run.py naive gbm

In [ ]:
# The transformer, with and without neighbours. On a GPU this uses the larger preset in neural.PRESETS["cuda"].
!python run.py neural

In [ ]:
# Zero-shot foundation models. Remove names to save time. On a GTX 1660 Ti the one-week variants take 10 to 35
# minutes each and the long-context variants 35 to 75 minutes each.
os.environ["SMOKESHIFT_FM"] = ",".join([
    "Chronos-2", "TimesFM", "Chronos-Bolt",
    "Chronos-2 (long context)", "TimesFM (long context)", "Chronos-Bolt (long context)",
])
!python run.py foundation

In [ ]:
# Optional: train each transformer five times with different seeds, to measure run-to-run spread.
# Skip this cell to keep the spread measured on the laptop (outputs/tables/seed_summary.csv).
!python run.py seeds

In [ ]:
# Ensembles, scoring, intervals, alerts, charts, report and dashboard data
!python run.py ensemble evaluate uncertainty alerts figures report dashboard

## 5. Look at the result

In [ ]:
import pandas as pd
scores = pd.read_csv("outputs/tables/scores_by_regime.csv")
scores.pivot(index="model", columns="slice", values="mae")[["All hours", "Calm at issue", "Smoke at issue"]].sort_values("All hours").round(3)

In [ ]:
from IPython.display import Image, display
for name in ["06_leaderboard.png", "07_error_by_horizon.png", "11_alerts.png"]:
    display(Image(f"outputs/figures/{name}"))

## 6. Take the results home

This zips everything small (tables, figures, report, dashboard). Unzip it over the project folder on your computer,
then commit and push.

In [ ]:
!zip -qr /content/smokeshift_results.zip outputs dashboard/data.js dashboard/index.html
from google.colab import files
files.download("/content/smokeshift_results.zip")

## Going further on a bigger GPU

Things a larger GPU makes practical, in order of how much they would add:

1. **A longer context for the trained models.** Three weeks of history improved every foundation model by about
   3%. The trees and the transformer still see one week (`config.CONTEXT`).
2. **A larger transformer.** Edit `PRESETS["cuda"]` in `src/smokeshift/neural.py` (for example `d_model` 256,
   `n_layers` 4, `epochs` 60).
3. **Fine-tuning a foundation model** on the training period, to separate what pretraining gives from what local
   data gives.
4. **A graph neural network** over the monitor network, as a learned replacement for the hand-built neighbour
   features.
5. **More seeds.** Lengthen `SEEDS` in `src/smokeshift/seeds.py` and run `python run.py seeds`.